<a href="https://colab.research.google.com/github/Miguel-EMC/recuperacion_de_informacion/blob/main/reranking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejercicio 10: Re-ranking

Objetivo: Implementar y evaluar un pipeline de Recuperación de Información en dos etapas, y analizar el impacto del re-ranking en la calidad del ranking.

## Parte 1: Preparación del corpus

- Cargar el corpus (documentos/pasajes).
- Cargar las consultas (queries).
- Cargar qrels (relevancia).

In [1]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd

/usr/local/lib/python3.12/dist-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
DATASET_NAME = "scifact"
DATA_DIR = "../data/beir_datasets"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET_NAME}.zip"
util.download_and_unzip(url, DATA_DIR)

'../data/beir_datasets/scifact'

In [3]:
dataset_path = DATA_DIR + "/" + DATASET_NAME
corpus, queries, qrels = GenericDataLoader(dataset_path).load(split="test")

  0%|          | 0/5183 [00:00<?, ?it/s]

In [4]:
df_corpus = (
    pd.DataFrame.from_dict(corpus, orient="index")
      .reset_index()
      .rename(columns={"index": "doc_id"})
)

df_corpus

,doc_id,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...
...,...,...,...
5178,195689316,BACKGROUND The main associations of body-mass ...,Body-mass index and cause-specific mortality i...
5179,195689757,A key aberrant biological difference between t...,Targeting metabolic remodeling in glioblastoma...
5180,196664003,A signaling pathway transmits information from...,Signaling architectures that transmit unidirec...
5181,198133135,AIMS Trabecular bone score (TBS) is a surrogat...,"Association between pre-diabetes, type 2 diabe..."


In [5]:
df_queries = (
    pd.DataFrame.from_dict(queries, orient="index", columns=["query"])
      .reset_index()
      .rename(columns={"index": "query_id"})
)

df_queries

,query_id,query
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...
...,...,...
295,1379,Women with a higher birth weight are more like...
296,1382,aPKCz causes tumour enhancement by affecting g...
297,1385,cSMAC formation enhances weak ligand signalling.
298,1389,mTORC2 regulates intracellular cysteine levels...


In [6]:
rows = []
for qid, docs in qrels.items():
    for doc_id, rel in docs.items():
        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "relevance": rel
        })

df_qrels = pd.DataFrame(rows)
df_qrels

,query_id,doc_id,relevance
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1
...,...,...,...
334,1379,17450673,1
335,1382,17755060,1
336,1385,306006,1
337,1389,23895668,1


In [7]:
# Elegimos una query cualquiera que tenga varios documentos relevantes
qid = "133"

print("Query:")
print(df_queries.loc[df_queries["query_id"] == qid, "query"].values[0])

print("\nDocumentos relevantes para esta query:")
df_qrels[(df_qrels["query_id"] == qid) & (df_qrels["relevance"] > 0)]

Query:
Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Documentos relevantes para esta query:


,query_id,doc_id,relevance
31,133,38485364,1
32,133,6969753,1
33,133,17934082,1
34,133,16280642,1
35,133,12640810,1


### Parte 2. Retrieval inicial (baseline)
- Implementar retrieval inicial con BM25
- Obtener métricas: Recall@10 nDCG@10

In [8]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from beir.retrieval.evaluation import EvaluateRetrieval

# Corpus
doc_ids = df_corpus["doc_id"].tolist()
texts   = (df_corpus["title"].fillna("") + " " + df_corpus["text"].fillna("")).tolist()

# BM25 index
k1, b = 1.5, 0.75
cv = CountVectorizer(lowercase=True)
tf = cv.fit_transform(texts)

n_docs   = tf.shape[0]
doc_lens = np.asarray(tf.sum(axis=1)).ravel()
avgdl    = doc_lens.mean()
df_term  = np.bincount(tf.nonzero()[1], minlength=tf.shape[1])
idf      = np.log((n_docs - df_term + 0.5) / (df_term + 0.5) + 1.0)

def bm25_retrieve(query: str, top_k: int = 100):
    q_tf  = cv.transform([query])
    q_idx = q_tf.nonzero()[1]
    scores = np.zeros(n_docs)
    for idx in q_idx:
        tf_col = np.asarray(tf[:, idx].todense()).ravel()
        norm   = tf_col + k1 * (1 - b + b * doc_lens / avgdl) + 1e-9
        scores += idf[idx] * (tf_col * (k1 + 1)) / norm
    top_i = np.argpartition(scores, -top_k)[-top_k:]
    top_i = top_i[np.argsort(scores[top_i])[::-1]]
    return {doc_ids[i]: float(scores[i]) for i in top_i}

TOP_K = 100
bm25_run = {}
for _, row in df_queries.iterrows():
    bm25_run[row["query_id"]] = bm25_retrieve(row["query"], top_k=TOP_K)

ndcg, _map, recall, _ = EvaluateRetrieval.evaluate(qrels, bm25_run, [10])
print("BM25 Baseline:")
print(f"  nDCG@10   = {ndcg['NDCG@10']:.4f}")
print(f"  Recall@10 = {recall['Recall@10']:.4f}")

BM25 Baseline:
  nDCG@10   = 0.6623
  Recall@10 = 0.7809


### Parte 3. Implementación del re-ranking cross-encoder
- Re-rankear los top-k candidatos para cada query.
- Identificar qué documentos cambian de posición en el top 10

In [ ]:
from sentence_transformers import CrossEncoder

model_ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

ce_run = {}
changes_log = []

for qid, cands in bm25_run.items():
    query     = queries[qid]
    cand_list = sorted(cands.items(), key=lambda x: x[1], reverse=True)

    pairs     = [[query, corpus[doc_id]["title"] + " " + corpus[doc_id]["text"]]
                 for doc_id, _ in cand_list]
    ce_scores = model_ce.predict(pairs, show_progress_bar=False)

    reranked  = sorted(zip([d for d, _ in cand_list], ce_scores),
                       key=lambda x: x[1], reverse=True)
    ce_run[qid] = {doc_id: float(s) for doc_id, s in reranked}

    # Cambios de posición en top-10
    bm25_top10 = [d for d, _ in cand_list[:10]]
    ce_top10   = [d for d, _ in reranked[:10]]
    for rank, doc_id in enumerate(ce_top10, 1):
        old_rank = bm25_top10.index(doc_id) + 1 if doc_id in bm25_top10 else None
        if old_rank != rank:
            changes_log.append({"query_id": qid, "doc_id": doc_id,
                                 "bm25_rank": old_rank, "ce_rank": rank})

df_changes = pd.DataFrame(changes_log)
print(f"Cambios de posición en top-10: {len(df_changes)}")
df_changes.head(10)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

### Parte 4. Implementación del re-ranking LTR
- Re-rankear los top-k candidatos para cada query.
- Identificar qué documentos cambian de posición en el top 10

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GroupShuffleSplit

def get_features(qid, doc_id, bm25_score, bm25_max):
    query  = queries[qid]
    q_toks = set(query.lower().split())
    doc    = corpus[doc_id]
    d_toks = (doc["title"] + " " + doc["text"]).lower().split()
    d_set  = set(d_toks)
    t_set  = set(doc["title"].lower().split())
    return [
        bm25_score / (bm25_max + 1e-9),               # f1: BM25 normalizado
        len(d_toks),                                    # f2: longitud doc
        len(q_toks & d_set) / (len(q_toks) + 1e-9),   # f3: cobertura query
        len(q_toks & t_set) / (len(q_toks) + 1e-9),   # f4: match título
    ]

rows_ltr = []
for qid, cands in bm25_run.items():
    bm25_max = max(cands.values())
    for doc_id, score in cands.items():
        rel   = qrels.get(qid, {}).get(doc_id, 0)
        feats = get_features(qid, doc_id, score, bm25_max)
        rows_ltr.append({"qid": qid, "doc_id": doc_id,
                          "f1": feats[0], "f2": feats[1],
                          "f3": feats[2], "f4": feats[3], "label": rel})

df_ltr = pd.DataFrame(rows_ltr)
FEATS  = ["f1", "f2", "f3", "f4"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(df_ltr, groups=df_ltr["qid"]))
df_train  = df_ltr.iloc[train_idx].reset_index(drop=True)
df_test   = df_ltr.iloc[test_idx].reset_index(drop=True)
test_qids = df_test["qid"].unique()

ltr_model = GradientBoostingRegressor(n_estimators=200, max_depth=4,
                                       learning_rate=0.1, random_state=42)
ltr_model.fit(df_train[FEATS], df_train["label"])

ltr_run = {}
for qid in test_qids:
    df_q = df_test[df_test["qid"] == qid]
    pred = ltr_model.predict(df_q[FEATS])
    ltr_run[qid] = dict(zip(df_q["doc_id"], pred.tolist()))

print(f"LTR entrenado — {len(test_qids)} queries de test")

### Parte 5. Evaluación post re-ranking
Calcular métricas:
- nDCG@10
- MAP
- Recall@10

In [ ]:
from beir.retrieval.evaluation import EvaluateRetrieval

test_q     = set(test_qids)
qrels_test = {q: v for q, v in qrels.items()    if q in test_q}
bm25_test  = {q: v for q, v in bm25_run.items() if q in test_q}
ce_test    = {q: v for q, v in ce_run.items()   if q in test_q}

summary = {}
for name, run in [("BM25", bm25_test), ("CrossEncoder", ce_test), ("LTR", ltr_run)]:
    ndcg, _map, recall, _ = EvaluateRetrieval.evaluate(qrels_test, run, [10])
    summary[name] = {
        "nDCG@10":   ndcg["NDCG@10"],
        "MAP@10":    _map["MAP@10"],
        "Recall@10": recall["Recall@10"],
    }

df_eval = pd.DataFrame(summary).T
print(df_eval.round(4))
df_eval.round(4).style.highlight_max(axis=0, color="lightgreen")